In [82]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

RANDOMSTATE = 42

In [84]:
df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")
# df["timestamp"] = pd.to_datetime(df["timestamp"])
# df.info()

In [ ]:
# for items in df["machine_status"]:
#     if items == "NORMAL":
#         print(items)

# def find_occurance(df, column, token):
#     num = 0
#     for status in df[column]:
#         if status == token:
#             num += 1
#     return num
# num_of_broken = find_occurance(df, 'machine_status', 'BROKEN')
# print(f'The number of BROKEN occurances are: {num_of_broken}')


df.drop(columns=["num", "timestamp", "machine_status"], axis=1,  inplace=True)


def return_columns_with_lots_of_NAN(df, limit):
    columns_to_remove = ''

    for column in df:
        column_length = len(df[column])
        nan = df[column].isna().sum()

        fraction = nan / column_length

        if fraction >= limit:
            columns_to_remove += (column + ', ')
    return columns_to_remove 
nan_string = return_columns_with_lots_of_NAN(df, 0.05)
print(f'Columns with more than the limit of NAN values: {nan_string}')


df.drop(columns=["sensor_15", "sensor_50", "sensor_51"], axis=1,  inplace=True)
df.fillna(method='ffill', inplace=True)
df.columns = df.columns.str.strip()


# split into training and testing data 
# standardScaler
# PCA 
# sns.heatmap(df)



# use standardscaler, and then PCA followed up with some of the recomended alternatives 
# identify more columns that have missing values and remove them??

# https://www.geeksforgeeks.org/principal-component-analysis-pca/

Columns with more than the limit of NAN values: sensor_15, sensor_50, sensor_51, 


In [ ]:
scaler = StandardScaler()


In [81]:
#create PCA model
pca = PCA(n_components = .99, svd_solver = 'full', random_state = RANDOMSTATE )
pca.fit(df)
print(f'Number of components after reduction: {pca.n_components_}')

ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

In [ ]:
# print(df.isnull().sum())
# df.head()
# df.describe()

# Select the first 3 columns, excluding "num" and "timestamp"
columns_to_plot = df.columns[:5]  # Get the first 3 columns

# Loop over the selected columns
# for column in columns_to_plot:
#     if column == "num" or column == "timestamp":
#         continue  # Skip "num" and "timestamp"
    
#     plt.figure(figsize=(22, 1), dpi=600)
#     plt.title(column)
#     plt.plot(df["num"], df[column])  # Plot against the "num" column
#     plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import Dense
from sklearn.metrics import mean_squared_error
import seaborn as sns

# Step 1: Load the data
df = pd.read_csv("C:/Users/andsa/Documents/ML/MLing/dataset/sensor.csv")

df.drop(columns=["machine_status", "sensor_15"], axis=1,  inplace=True)
df.fillna(method='ffill', inplace=True)
df.columns = df.columns.str.strip()

# Convert 'timestamp' to datetime and remove it
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.drop(columns=["timestamp"])

# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(df)

# Step 2: Build the Autoencoder Model
# Define the autoencoder model architecture
autoencoder = Sequential()
autoencoder.add(Dense(64, input_dim=data_scaled.shape[1], activation='relu'))
autoencoder.add(Dense(32, activation='relu'))
autoencoder.add(Dense(16, activation='relu'))
autoencoder.add(Dense(32, activation='relu'))
autoencoder.add(Dense(64, activation='relu'))
autoencoder.add(Dense(data_scaled.shape[1], activation='sigmoid'))  # Output layer

# Compile the model
autoencoder.compile(optimizer='adam', loss='mean_squared_error')

# Step 3: Train the Autoencoder
# Train the autoencoder on the normal data (since it's unsupervised, we don't need labels)
autoencoder.fit(data_scaled, data_scaled, epochs=50, batch_size=256, shuffle=True)

# Step 4: Use the trained model to detect anomalies
# Compute the reconstruction error for each data point
reconstructed = autoencoder.predict(data_scaled)
reconstruction_error = mean_squared_error(data_scaled, reconstructed, multioutput='raw_values')

# Set a threshold for anomaly detection (e.g., based on the distribution of the error)
threshold = np.percentile(reconstruction_error, 95)  # 95th percentile as the threshold for anomaly

# Mark anomalies where the reconstruction error exceeds the threshold
anomalies = reconstruction_error > threshold

# Step 5: Visualize the results
plt.figure(figsize=(10,6))
plt.plot(df.index, reconstruction_error, label='Reconstruction Error')
plt.axhline(y=threshold, color='r', linestyle='--', label='Anomaly Threshold')
plt.scatter(df.index[anomalies], reconstruction_error[anomalies], color='r', label='Anomalies')
plt.title('Reconstruction Error and Anomalies')
plt.xlabel('Index')
plt.ylabel('Reconstruction Error')
plt.legend()
plt.show()

# Step 6: Output the anomalies
df['anomaly'] = anomalies
print(df[df['anomaly'] == True])  # Display rows identified as anomalies
